In [ ]:
import smrt
from smrt.inputs.make_medium import make_ice_column
from smrt import make_snowpack, sensor_list, PSU, make_model, make_atmosphere
import numpy as np
import pandas as pd
from smrt.permittivity.generic_mixing_formula import polder_van_santen
from smrt.permittivity.ice import ice_permittivity_maetzler06
import csv

def seawater_permittivity_meissner_wentz(freq, sst_in, salinity):
    """Computes seawater dielectric constant using Meissner and Wentz."""
    f0 = 17.97510
    x = [5.7230e+00, 2.2379e-02, -7.1237e-04, 5.0478e+00, -7.0315e-02, 6.0059e-04, 
         3.6143e+00, 2.8841e-02, 1.3652e-01, 1.4825e-03, 2.4166e-04]
    z = [-3.56417e-03, 4.74868e-06, 1.15574e-05, 2.39357e-03, -3.13530e-05, 
         2.52477e-07, -6.28908e-03, 1.76032e-04, -9.22144e-05, -1.99723e-02, 
         1.81176e-04, -2.04265e-03, 1.57883e-04]
    a0coef = [-0.33330E-02, 4.74868e-06, 0.0e+00]
    b1coef = [0.23232E-02, -0.79208E-04, 0.36764E-05, -0.35594E-06, 0.89795E-08]

    sst = np.array(sst_in)
    salinity = np.array(salinity)
    sst[sst <= -30.16] = -30.16
    sst2, sst3, sst4, salinity2 = sst**2, sst**3, sst**4, salinity**2

    e0 = (3.70886e4 - 8.2168e1 * sst) / (4.21854e2 + sst)
    e1 = x[0] + x[1] * sst + x[2] * sst2
    n1 = (45.00 + sst) / (x[3] + x[4] * sst + x[5] * sst2)
    e2 = x[6] + x[7] * sst
    n2 = (45.00 + sst) / (x[8] + x[9] * sst + x[10] * sst2)

    sig35 = 2.903602 + 8.60700e-2 * sst + 4.738817e-4 * sst2 - 2.9910e-6 * sst3 + 4.3047e-9 * sst4
    r15 = salinity * (37.5109 + 5.45216 * salinity + 1.4409e-2 * salinity2) / (1004.75 + 182.283 * salinity + salinity2)
    alpha0 = (6.9431 + 3.2841 * salinity - 9.9486e-2 * salinity2) / (84.850 + 69.024 * salinity + salinity2)
    alpha1 = 49.843 - 0.2276 * salinity + 0.198e-2 * salinity2
    rtr15 = 1.0 + (sst - 15.0) * alpha0 / (alpha1 + sst)
    sig = sig35 * r15 * rtr15

    a0 = np.exp(a0coef[0] * salinity + a0coef[1] * salinity2 + a0coef[2] * salinity * sst)
    e0s = a0 * e0

    b1 = np.ones_like(sst)
    mask = sst <= 30
    for m in [mask, ~mask]:
        b1[m] += salinity[m] * sum(b1coef[i] * sst[m]**i for i in range(5))
    n1s = n1 * b1

    a1 = np.exp(z[6] * salinity + z[7] * salinity2 + z[8] * salinity * sst)
    e1s = e1 * a1
    b2 = 1.0 + salinity * (z[9] + 0.5 * z[10] * (sst + 30))
    n2s = n2 * b2
    a2 = 1.0 + salinity * (z[11] + z[12] * sst)
    e2s = e2 * a2

    epsr = (e0s - e1s) / (1.0 - 1j * freq / n1s) + (e1s - e2s) / (1.0 - 1j * freq / n2s) + e2s + 1j * sig * f0 / freq
    return epsr

def extract_parameters_with_min_max(sheet_data, layer_keywords, parameter_to_test):
    extracted_parameters = {}
    for layer_prefix in layer_keywords:
        filtered_data = sheet_data[sheet_data['Parameter'].str.contains(layer_prefix, na=False)]
        layer_data = {}
        for _, row in filtered_data.iterrows():
            param_name = row['Parameter']
            if param_name == parameter_to_test:
                layer_data[param_name] = {'Min': row['Min'], 'Max': row['Max'], 'Value': row['Value REF']}
            else:
                layer_data[param_name] = {'Value': row['Value REF']}
        extracted_parameters[layer_prefix] = layer_data
    return extracted_parameters

def create_snowpack_and_run_model(layer_parameters, frequency, parameter_to_test, atmos19, atmos37, N=21, output_file="results.csv"):
    min_val = max_val = None
    for layer_key in ['SP', 'DH', 'SI_1', 'SI_2']:
        if parameter_to_test in layer_parameters[layer_key] and isinstance(layer_parameters[layer_key][parameter_to_test], dict):
            min_val = layer_parameters[layer_key][parameter_to_test].get('Min')
            max_val = layer_parameters[layer_key][parameter_to_test].get('Max')
            break

    if min_val is None or max_val is None or min_val == max_val:
        raise ValueError(f"Invalid Min/Max values for '{parameter_to_test}'")

    test_values = np.linspace(float(min_val), float(max_val), N)
    results = []

    for value in test_values:
        for layer_key in ['SP', 'DH', 'SI_1', 'SI_2']:
            if parameter_to_test in layer_parameters[layer_key]:
                layer_parameters[layer_key][parameter_to_test]['Value'] = float(value)
                break

        sp_params = {k: v['Value'] for k, v in layer_parameters['SP'].items()}
        dh_params = {k: v['Value'] for k, v in layer_parameters['DH'].items()}
        si1_params = {k: v['Value'] for k, v in layer_parameters['SI_1'].items()}
        si2_params = {k: v['Value'] for k, v in layer_parameters['SI_2'].items()}

        density_ice = 917
        ssa_sp = 3 / (sp_params['Radius_SP'] * density_ice)
        porod_length_sp = 4 * (1 - sp_params['Density_SP'] / density_ice) / (ssa_sp * density_ice)
        ssa_dh = 3 / (dh_params['Radius_DH'] * density_ice)
        porod_length_dh = 4 * (1 - dh_params['Density_DH'] / density_ice) / (ssa_dh * density_ice)

        eps_saline_water_sp = seawater_permittivity_meissner_wentz(frequency, sp_params['Temperature_SP'] - 273.15, sp_params['Salinity_SP'])
        eps_ice_sp = ice_permittivity_maetzler06(frequency*10**9, sp_params['Temperature_SP'])
        eps_saltedwater_ice_pvs_reverse_sp = polder_van_santen(sp_params['Fraction_volume_water_SP'], eps_ice_sp, eps_saline_water_sp)

        eps_saline_water_dh = seawater_permittivity_meissner_wentz(frequency, dh_params['Temperature_DH'] - 273.15, dh_params['Salinity_DH'])
        eps_ice_dh = ice_permittivity_maetzler06(frequency*10**9, dh_params['Temperature_DH'])
        eps_saltedwater_ice_pvs_reverse_dh = polder_van_santen(dh_params['Fraction_volume_water_DH'], eps_ice_dh, eps_saline_water_dh)

        sp_top = make_snowpack(
            thickness=[sp_params['Thickness_SP']], density=[sp_params['Density_SP']],
            temperature=[sp_params['Temperature_SP']], microstructure_model="unified_scaled_exponential",
            porod_length=porod_length_sp, polydispersity=0.7, radius=[sp_params['Radius_SP']],
            ice_permittivity_model=eps_saltedwater_ice_pvs_reverse_sp)

        dh_layer = make_snowpack(
            thickness=[dh_params['Thickness_DH']], density=[dh_params['Density_DH']],
            temperature=[dh_params['Temperature_DH']], microstructure_model="unified_scaled_exponential",
            porod_length=porod_length_dh, polydispersity=1.5, radius=[dh_params['Radius_DH']],
            ice_permittivity_model=eps_saltedwater_ice_pvs_reverse_dh)

        ic1 = make_ice_column('firstyear', thickness=[si1_params["Thickness_SI_1"]],
            microstructure_model='sticky_hard_spheres', stickiness=100,
            density=[si1_params["Density_SI_1"]], temperature=[si1_params["Temperature_SI_1"]],
            salinity=[si1_params["Salinity_SI_1"] * PSU], radius=[si1_params["Radius_SI_1"]], add_water_substrate=False)

        ic2 = make_ice_column('firstyear', thickness=[si2_params["Thickness_SI_2"]],
            microstructure_model='sticky_hard_spheres', stickiness=100,
            density=[si2_params["Density_SI_2"]], temperature=[si2_params["Temperature_SI_2"]],
            salinity=[si2_params["Salinity_SI_2"] * PSU], radius=[si2_params["Radius_SI_2"]], add_water_substrate=True)

        total_snowpack = (atmos37 if frequency == 36.5 else atmos19) + sp_top + dh_layer + ic1 + ic2
        sensor = sensor_list.amsr2('37' if frequency == 36.5 else '19')
        snowpack_model = make_model("iba", "dort", rtsolver_options=dict(n_max_stream=128))
        result_seaice = snowpack_model.run(sensor, total_snowpack)

        results.append({"Parameter": parameter_to_test, "Value": value, "TbH": result_seaice.TbH(), "TbV": result_seaice.TbV()})

    with open(output_file, mode='w', newline='') as file:
        writer = csv.DictWriter(file, fieldnames=["Parameter", "Value", "TbH", "TbV"])
        writer.writeheader()
        writer.writerows(results)
    return results

def load_tb_data(filepath, max_theta=75):
    df = pd.read_csv(filepath, delim_whitespace=True, na_values='*****', comment='$', engine='python')
    df.columns = df.columns.str.strip().str.rstrip(',')
    df = df[df['theta(deg)'] <= max_theta]
    return {'theta': df['theta(deg)'].to_list(), 'tb_down': df['TbDown(K)'].to_list(),
            'tb_up': df['TbUp(k)'].to_list(), 'transmittance': np.exp(-df['tau(neper)']).to_list()}

# Load atmosphere data
file_paths = {'19': '[..]/TbAtmo_19_1_9.dat', '37': '[..]/TbAtmo_37_1_9.dat', '89': '[..]/TbAtmo_89_1_9.dat'}
data = {k: load_tb_data(v) for k, v in file_paths.items()}
atmos = {k: make_atmosphere("simple_atmosphere", **v) for k, v in data.items()}

# Load parameters and run sensitivity analysis
file_path = '[..]/sensitivity_test_single_column_snowpack_winter.xlsx'
sheet_data = pd.read_excel(file_path, sheet_name="Sheet1")
frequencies = {"37": 36.5, "19": 18.7}
parameters_to_test = ["Temperature_SP", "Radius_SP", "Salinity_SP", "Fraction_volume_water_SP",
    "Thickness_SP", "Density_SP", "Temperature_DH", "Radius_DH", "Salinity_DH", 
    "Fraction_volume_water_DH", "Thickness_DH", "Density_DH", "Thickness_SI_1", 
    "Temperature_SI_1", "Salinity_SI_1"]
layer_keywords = ["SP", "DH", "SI_1", "SI_2"]
base_output_path = "[..]/Results_sensitivity_analysis/Round_paper"

for parameter in parameters_to_test:
    layer_parameters = extract_parameters_with_min_max(sheet_data, layer_keywords, parameter)
    for freq_label, freq_value in frequencies.items():
        output_file = f"{base_output_path}/results_sensitivity_winter_{freq_label}_{parameter}.csv"
        create_snowpack_and_run_model(layer_parameters, freq_value, parameter, atmos['19'], atmos['37'], 21, output_file)